# 04 — Procesamiento Distribuido con Apache Spark (Cloud — GCS + Dataproc YARN)

Versión cloud del notebook Spark. Los CSVs se leen directamente desde **GCS** y Spark intenta conectarse al cluster Dataproc via **YARN**. Si el notebook se ejecuta fuera del cluster (desde el PC local), Spark cae a modo local[4] automáticamente.

**Flujo:** GCS raw/ (2017–2025) → Spark YARN en Dataproc (o local[4] como fallback) → resultados por era COVID → BigQuery

| # | Operación | Descripción | Tabla BigQuery |
|---|-----------|-------------|----------------|
| 1 | **Create** | Arrestos por año y era COVID — tendencia de efectividad policial | `arrests_by_year` |
| 2 | **Read**   | Crímenes por community area — comparativa geográfica entre eras | `crimes_by_district` |
| 3 | **Read**   | Tendencia mensual 2017–2025 — estacionalidad por era COVID | `monthly_trend` |
| 4 | **Update** | Añadir `is_weekend` + `covid_era` — patrones fin de semana por COVID | `weekend_analysis` |
| 5 | **Delete** | Filtrar registros inválidos — validar integridad por era | — (limpieza) |

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'pandas-gbq', '--quiet'], check=True)
print('Dependencias cloud OK')

Dependencias cloud OK


In [2]:
import os
import time
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    LongType, DoubleType
)

PROJECT_ID = 'my-first-project-492901'
DATASET_ID = 'chicago_crimes_results'
GCS_BUCKET = 'gs://big-data-proyecto-parcial/raw'

YEARS_ANALYSIS = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

def to_bigquery(spark_df, table_name: str, if_exists: str = 'replace') -> None:
    pdf = spark_df.toPandas()
    pdf.to_gbq(
        destination_table=f'{DATASET_ID}.{table_name}',
        project_id=PROJECT_ID,
        if_exists=if_exists,
        progress_bar=False,
    )
    print(f'  → BigQuery: {PROJECT_ID}.{DATASET_ID}.{table_name}  ({pdf.shape[0]:,} filas)')

# ── Detectar si estamos en Dataproc (YARN) o local ────────────────────────────
YARN_MASTER = os.environ.get('YARN_CONF_DIR') is not None  # True dentro de Dataproc
spark_master = 'yarn' if YARN_MASTER else 'local[4]'
print(f'Spark master: {spark_master}')

builder = (
    SparkSession.builder
    .appName('ChicagoCrimes-COVID-CRUD-Cloud')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '16')
    .config('spark.ui.showConsoleProgress', 'false')
)
if YARN_MASTER:
    builder = (
        builder
        .config('spark.submit.deployMode', 'client')
        .config('spark.executor.memory', '6g')
        .config('spark.executor.cores', '2')
    )

spark = builder.getOrCreate()
print(f'Spark {spark.version} — modo {spark_master}')

Spark master: local[4]


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/02 02:19:11 INFO SparkEnv: Registering MapOutputTracker
26/05/02 02:19:11 INFO SparkEnv: Registering BlockManagerMaster
26/05/02 02:19:11 INFO SparkEnv: Registering BlockManagerMasterHeartbeat


26/05/02 02:19:11 INFO SparkEnv: Registering OutputCommitCoordinator


Spark 3.5.3 — modo local[4]


In [3]:
# ── Schema explícito + carga desde GCS ───────────────────────────────────────
schema = StructType([
    StructField('unique_key',           LongType(),    True),
    StructField('case_number',          StringType(),  True),
    StructField('date',                 StringType(),  True),
    StructField('block',                StringType(),  True),
    StructField('iucr',                 StringType(),  True),
    StructField('primary_type',         StringType(),  True),
    StructField('description',          StringType(),  True),
    StructField('location_description', StringType(),  True),
    StructField('arrest',               StringType(),  True),
    StructField('domestic',             StringType(),  True),
    StructField('beat',                 IntegerType(), True),
    StructField('district',             IntegerType(), True),
    StructField('ward',                 IntegerType(), True),
    StructField('community_area',       IntegerType(), True),
    StructField('fbi_code',             StringType(),  True),
    StructField('x_coordinate',         LongType(),    True),
    StructField('y_coordinate',         LongType(),    True),
    StructField('year',                 IntegerType(), True),
    StructField('updated_on',           StringType(),  True),
    StructField('latitude',             DoubleType(),  True),
    StructField('longitude',            DoubleType(),  True),
    StructField('location',             StringType(),  True),
])

era_expr = F.when(F.col('year') <= 2019, 'PRE') \
            .when(F.col('year') <= 2022, 'DURANTE') \
            .otherwise('POST')

t0 = time.time()

if YARN_MASTER:
    # En Dataproc: GCS connector nativo disponible — leer directo
    files_gs = [f'{GCS_BUCKET}/Chicago_Crimes_{y}.csv' for y in YEARS_ANALYSIS]
    df_raw = (
        spark.read
             .option('header', 'true')
             .option('nullValue', '')
             .schema(schema)
             .csv(files_gs)
    )
else:
    # En PC local: Spark no tiene GCS connector — usar gcsfs/pandas para leer y convertir
    import gcsfs
    import pandas as _pd
    print('Modo local: leyendo desde GCS via gcsfs...')
    fs = gcsfs.GCSFileSystem(token='google_default')
    bucket_path = GCS_BUCKET.replace('gs://', '')
    dfs = []
    for y in YEARS_ANALYSIS:
        path = f'{bucket_path}/Chicago_Crimes_{y}.csv'
        with fs.open(path, 'rb') as f_csv:
            dfs.append(_pd.read_csv(f_csv, dtype=str, low_memory=False))
    pdf = _pd.concat(dfs, ignore_index=True)
    # Convertir tipos numéricos
    for col_name in ['unique_key','beat','district','ward','community_area',
                     'x_coordinate','y_coordinate','year']:
        pdf[col_name] = _pd.to_numeric(pdf[col_name], errors='coerce')
    for col_name in ['latitude','longitude']:
        pdf[col_name] = _pd.to_numeric(pdf[col_name], errors='coerce')
    print(f'  Leídos {len(pdf):,} registros con gcsfs en {time.time()-t0:.1f}s')
    df_raw = spark.createDataFrame(pdf)

df = (
    df_raw
    .withColumn('date',       F.to_timestamp('date'))
    .withColumn('updated_on', F.to_timestamp('updated_on'))
    .withColumn('arrest',     F.lower(F.col('arrest')) == 'true')
    .withColumn('domestic',   F.lower(F.col('domestic')) == 'true')
    .withColumn('covid_era',  era_expr)
)
df.cache()
total = df.count()
print(f'Fuente:             GCS — {GCS_BUCKET}/')
print(f'Total de registros: {total:,}  ({time.time()-t0:.1f}s)')
print(f'Años cargados:      {YEARS_ANALYSIS}')

df.createOrReplaceTempView('crimes')
print('Vista SQL "crimes" disponible.')

Modo local: leyendo desde GCS via gcsfs...


  Leídos 2,072,943 registros con gcsfs en 23.5s


26/05/02 02:22:07 WARN TaskSetManager: Stage 0 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


26/05/02 02:22:26 WARN TaskSetManager: Stage 1 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


Fuente:             GCS — gs://big-data-proyecto-parcial/raw/
Total de registros: 2,072,943  (189.3s)
Años cargados:      [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Vista SQL "crimes" disponible.


---
## CRUD 1 — CREATE: Arrestos por año y era COVID

In [4]:
arrests_by_year = spark.sql("""
    SELECT
        year,
        covid_era,
        COUNT(*)                                    AS total_crimes,
        SUM(CAST(arrest AS INT))                    AS total_arrests,
        ROUND(AVG(CAST(arrest AS INT)) * 100, 2)    AS arrest_rate_pct
    FROM crimes
    WHERE year IS NOT NULL
    GROUP BY year, covid_era
    ORDER BY year
""")

arrests_by_year.show(30)

print('Tasa de arresto promedio por era:')
arrests_by_year.groupBy('covid_era') \
    .agg(
        F.round(F.sum('total_crimes'), 0).alias('total_crimes'),
        F.round(F.avg('arrest_rate_pct'), 2).alias('avg_arrest_rate_pct'),
    ) \
    .orderBy('covid_era') \
    .show()

to_bigquery(arrests_by_year, 'arrests_by_year')

26/05/02 02:22:30 WARN TaskSetManager: Stage 4 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


+----+---------+------------+-------------+---------------+
|year|covid_era|total_crimes|total_arrests|arrest_rate_pct|
+----+---------+------------+-------------+---------------+
|2017|      PRE|      269214|        52670|          19.56|
|2018|      PRE|      269070|        53901|          20.03|
|2019|      PRE|      261555|        56256|          21.51|
|2020|  DURANTE|      212522|        34158|          16.07|
|2021|  DURANTE|      209406|        26558|          12.68|
|2022|  DURANTE|      239655|        28074|          11.71|
|2023|     POST|      262756|        31843|          12.12|
|2024|     POST|      256305|        34502|          13.46|
|2025|     POST|       92460|        15117|          16.35|
+----+---------+------------+-------------+---------------+

Tasa de arresto promedio por era:


26/05/02 02:22:35 WARN TaskSetManager: Stage 7 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


+---------+------------+-------------------+
|covid_era|total_crimes|avg_arrest_rate_pct|
+---------+------------+-------------------+
|  DURANTE|      661583|              13.49|
|     POST|      611521|              13.98|
|      PRE|      799839|              20.37|
+---------+------------+-------------------+



26/05/02 02:22:37 WARN TaskSetManager: Stage 19 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


/tmp/ipykernel_42290/1830428804.py:19: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  pdf.to_gbq(


  → BigQuery: my-first-project-492901.chicago_crimes_results.arrests_by_year  (9 filas)


---
## CRUD 2 — READ: Crímenes por community area — comparativa geográfica entre eras

In [5]:
crimes_by_area = spark.sql("""
    SELECT
        community_area                              AS district,
        covid_era,
        COUNT(*)                                    AS total_crimes,
        SUM(CAST(arrest AS INT))                    AS total_arrests,
        ROUND(AVG(CAST(arrest AS INT)) * 100, 2)    AS arrest_rate_pct,
        COUNT(DISTINCT primary_type)                AS distinct_crime_types,
        SUM(CAST(domestic AS INT))                  AS domestic_incidents
    FROM crimes
    WHERE community_area IS NOT NULL
    GROUP BY community_area, covid_era
    ORDER BY community_area, covid_era
""")

print('Top 5 áreas comunitarias con más crímenes por era:')
for era in ['PRE', 'DURANTE', 'POST']:
    print(f'\n  Era {era}:')
    crimes_by_area.filter(F.col('covid_era') == era) \
        .orderBy(F.col('total_crimes').desc()) \
        .select('district', 'total_crimes', 'arrest_rate_pct') \
        .show(5)

to_bigquery(crimes_by_area, 'crimes_by_district')

Top 5 áreas comunitarias con más crímenes por era:

  Era PRE:


26/05/02 02:22:43 WARN TaskSetManager: Stage 27 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


+--------+------------+---------------+
|district|total_crimes|arrest_rate_pct|
+--------+------------+---------------+
|    25.0|       45474|          25.36|
|     8.0|       38067|          16.99|
|    32.0|       32144|          15.74|
|    28.0|       27944|          15.61|
|    29.0|       27611|          38.92|
+--------+------------+---------------+
only showing top 5 rows


  Era DURANTE:


26/05/02 02:22:45 WARN TaskSetManager: Stage 30 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


+--------+------------+---------------+
|district|total_crimes|arrest_rate_pct|
+--------+------------+---------------+
|    25.0|       36802|          16.79|
|     8.0|       26350|           13.4|
|    43.0|       23617|          10.76|
|    28.0|       23000|          10.21|
|    29.0|       20688|          24.04|
+--------+------------+---------------+
only showing top 5 rows


  Era POST:


26/05/02 02:22:47 WARN TaskSetManager: Stage 33 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


+--------+------------+---------------+
|district|total_crimes|arrest_rate_pct|
+--------+------------+---------------+
|    25.0|       30504|          16.97|
|     8.0|       26511|           15.5|
|    28.0|       25099|           9.92|
|    32.0|       21277|          20.55|
|    43.0|       20405|          10.31|
+--------+------------+---------------+
only showing top 5 rows



26/05/02 02:22:50 WARN TaskSetManager: Stage 36 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


/tmp/ipykernel_42290/1830428804.py:19: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  pdf.to_gbq(


  → BigQuery: my-first-project-492901.chicago_crimes_results.crimes_by_district  (233 filas)


---
## CRUD 3 — READ: Tendencia mensual 2017–2025

In [6]:
monthly_trend = spark.sql("""
    SELECT
        year,
        covid_era,
        MONTH(date)                                 AS month,
        COUNT(*)                                    AS total_crimes,
        SUM(CAST(arrest AS INT))                    AS arrests,
        ROUND(AVG(CAST(arrest AS INT)) * 100, 2)    AS arrest_rate_pct
    FROM crimes
    WHERE date IS NOT NULL AND year IS NOT NULL
    GROUP BY year, covid_era, MONTH(date)
    ORDER BY year, month
""")

print(f'Puntos temporales (año×mes): {monthly_trend.count()}')

monthly_trend.createOrReplaceTempView('monthly')
seasonal = spark.sql("""
    SELECT
        covid_era,
        CASE WHEN month IN (6,7,8) THEN 'VERANO' ELSE 'RESTO' END AS season,
        ROUND(AVG(total_crimes), 0) AS avg_monthly_crimes
    FROM monthly
    GROUP BY covid_era, CASE WHEN month IN (6,7,8) THEN 'VERANO' ELSE 'RESTO' END
    ORDER BY covid_era, season
""")
print('Criminalidad promedio mensual — verano vs resto por era:')
seasonal.show()

to_bigquery(monthly_trend, 'monthly_trend')

26/05/02 02:22:56 WARN TaskSetManager: Stage 49 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


Puntos temporales (año×mes): 102
Criminalidad promedio mensual — verano vs resto por era:


26/05/02 02:22:58 WARN TaskSetManager: Stage 55 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


+---------+------+------------------+
|covid_era|season|avg_monthly_crimes|
+---------+------+------------------+
|  DURANTE| RESTO|           17851.0|
|  DURANTE|VERANO|           19956.0|
|     POST| RESTO|           20189.0|
|     POST|VERANO|           21023.0|
|      PRE| RESTO|           21415.0|
|      PRE|VERANO|           24625.0|
+---------+------+------------------+



26/05/02 02:23:00 WARN TaskSetManager: Stage 61 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


/tmp/ipykernel_42290/1830428804.py:19: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  pdf.to_gbq(


  → BigQuery: my-first-project-492901.chicago_crimes_results.monthly_trend  (102 filas)


---
## CRUD 4 — UPDATE: Añadir `is_weekend` + análisis por día de semana cruzado con era COVID

In [7]:
df_enriched = (
    df
    .withColumn('is_weekend', F.dayofweek('date').isin([1, 7]))
    .withColumn('day_name',   F.date_format('date', 'EEEE'))
)
df_enriched.createOrReplaceTempView('crimes')

weekend_analysis = spark.sql("""
    SELECT
        covid_era,
        is_weekend,
        COUNT(*)                                            AS total_crimes,
        SUM(CAST(arrest AS INT))                            AS total_arrests,
        ROUND(AVG(CAST(arrest AS INT)) * 100, 2)            AS arrest_rate_pct,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY covid_era), 2) AS pct_of_era_total
    FROM crimes
    WHERE is_weekend IS NOT NULL
    GROUP BY covid_era, is_weekend
    ORDER BY covid_era, is_weekend DESC
""")

print('Fin de semana vs días de semana por era COVID:')
weekend_analysis.show()

to_bigquery(weekend_analysis, 'weekend_analysis')

Fin de semana vs días de semana por era COVID:


26/05/02 02:23:06 WARN TaskSetManager: Stage 69 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


+---------+----------+------------+-------------+---------------+----------------+
|covid_era|is_weekend|total_crimes|total_arrests|arrest_rate_pct|pct_of_era_total|
+---------+----------+------------+-------------+---------------+----------------+
|  DURANTE|      true|      190638|        26393|          13.84|           28.82|
|  DURANTE|     false|      470945|        62397|          13.25|           71.18|
|     POST|      true|      176901|        23446|          13.25|           28.93|
|     POST|     false|      434620|        58016|          13.35|           71.07|
|      PRE|      true|      228097|        47333|          20.75|           28.52|
|      PRE|     false|      571742|       115494|           20.2|           71.48|
+---------+----------+------------+-------------+---------------+----------------+



26/05/02 02:23:15 WARN TaskSetManager: Stage 75 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


/tmp/ipykernel_42290/1830428804.py:19: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  pdf.to_gbq(


  → BigQuery: my-first-project-492901.chicago_crimes_results.weekend_analysis  (6 filas)


---
## CRUD 5 — DELETE: Filtrar registros inválidos y verificar integridad por era

In [8]:
invalid = df_enriched.filter(
    F.col('primary_type').isNull() |
    (F.trim(F.col('primary_type')) == '') |
    F.col('unique_key').isNull()
)

invalid_by_era = (
    invalid.groupBy('covid_era')
           .count()
           .withColumnRenamed('count', 'invalid_records')
)
total_by_era = (
    df_enriched.groupBy('covid_era')
               .count()
               .withColumnRenamed('count', 'total')
)
report = total_by_era.join(invalid_by_era, on='covid_era', how='left').fillna(0)
report = report.withColumn('pct_invalid',
    F.round(F.col('invalid_records') / F.col('total') * 100, 4))

print(f'Registros totales:   {total:,}')
print('Registros inválidos por era:')
report.orderBy('covid_era').show()

df_valid = df_enriched.filter(
    F.col('primary_type').isNotNull() &
    (F.trim(F.col('primary_type')) != '') &
    F.col('unique_key').isNotNull()
)
n_valid = df_valid.count()
print(f'Registros válidos:   {n_valid:,}')

spark.stop()
print('SparkSession cerrada. Resultados guardados en BigQuery.')

Registros totales:   2,072,943
Registros inválidos por era:


26/05/02 02:23:38 WARN TaskSetManager: Stage 88 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


26/05/02 02:23:43 WARN TaskSetManager: Stage 89 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


+---------+------+---------------+-----------+
|covid_era| total|invalid_records|pct_invalid|
+---------+------+---------------+-----------+
|  DURANTE|661583|              0|        0.0|
|     POST|611521|              0|        0.0|
|      PRE|799839|              0|        0.0|
+---------+------+---------------+-----------+



26/05/02 02:23:52 WARN TaskSetManager: Stage 93 contains a task of very large size (174380 KiB). The maximum recommended task size is 1000 KiB.


Registros válidos:   2,072,943


SparkSession cerrada. Resultados guardados en BigQuery.
